# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fatima-zehra5/ML-internhip/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method:** Logistic Regression classification.

I use logistic regression because the task is binary classification: predict whether a content item is likely to experience an impression decline. The model is simple, interpretable, and produces a probability that can support prioritization. This makes it a useful first modeling method before considering more complex models.

The goal is decision-support rather than claiming that the model proves why impressions decline.

In [1]:
# ML-08 — Method setup and reproducible imports

!pip -q install -U huggingface_hub duckdb scikit-learn pandas pyarrow

import duckdb
import pandas as pd
import numpy as np

from google.colab import userdata
from huggingface_hub import login, hf_hub_download

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("Libraries loaded.")
print("Method: Logistic Regression classification.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.2/793.2 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 11.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.5 which is incompatible.
Libraries loaded.
Method: Logistic Regression classification.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Split design:** I use a reproducible train/test split with stratification so both classes are represented in the evaluation data.

The split is used for model development and comparison on the available development slice. The target is kept separate from the input features to avoid direct label leakage. The evaluation is treated as directional decision-support rather than a claim about future production performance.

In [2]:
# Load the March 2026 development partition

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

login(token=HF_TOKEN, add_to_git_credential=False)

march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset"
)

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE TABLE fact_content_daily_performance AS
SELECT *
FROM read_parquet('{march_path}')
""")

print("March 2026 data loaded.")

# Inspect available columns
columns_df = con.sql("""
DESCRIBE fact_content_daily_performance
""").df()

display(columns_df)

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March 2026 data loaded.


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [8]:
# Inspect the key columns needed for the modeling lane

available_columns = set(
    columns_df["column_name"].astype(str).str.lower()
)

required_candidates = {
    "client_hash_id",
    "content_hash_id",
    "report_date",
    "gsc_impressions"
}

missing = sorted(required_candidates - available_columns)

print("Required modeling columns:", sorted(required_candidates))
print("Missing columns:", missing)

if missing:
    raise ValueError(
        f"Required columns are missing from the March dataset: {missing}"
    )

print("All required modeling columns are available.")

Required modeling columns: ['client_hash_id', 'content_hash_id', 'gsc_impressions', 'report_date']
Missing columns: []
All required modeling columns are available.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

**Baseline:** The simple baseline predicts the majority class for every observation.

**Model:** Logistic Regression uses pre-outcome impression features to estimate the probability of an impression decline.

I compare both approaches on the same held-out test set using accuracy, precision, recall, and F1. F1 is especially useful here because the positive class is the action-relevant decline class and accuracy alone can be misleading when classes are uneven.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Build a compact content-by-client-by-day table.
# This keeps the modeling unit aligned with the data contract.

daily = con.sql("""
SELECT
    client_hash_id,
    content_hash_id AS content_id, -- Corrected column name
    CAST(report_date AS DATE) AS report_date,
    SUM(gsc_impressions) AS impressions -- Corrected column name
FROM fact_content_daily_performance
GROUP BY
    client_hash_id,
    content_hash_id,
    CAST(report_date AS DATE)
ORDER BY
    client_hash_id,
    content_hash_id,
    report_date
""").df()

print("Daily modeling table shape:", daily.shape)

display(daily.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Daily modeling table shape: (9841378, 4)


,client_hash_id,content_id,report_date,impressions
0,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,2026-03-01,0.0
1,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,2026-03-02,0.0
2,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,2026-03-03,0.0
3,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,2026-03-04,0.0
4,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,2026-03-05,0.0


In [11]:
# Create prior-7-day and next-7-day impression windows.
# The target uses the future window, while features use prior information.

daily = daily.sort_values(
    ["client_hash_id", "content_id", "report_date"]
).copy()

group_cols = ["client_hash_id", "content_id"]

daily["prior_7d_impressions"] = (
    daily.groupby(group_cols)["impressions"]
    .transform(
        lambda s: s.shift(1).rolling(7, min_periods=3).sum()
    )
)

daily["future_7d_impressions"] = (
    daily.iloc[::-1]
    .groupby(group_cols)["impressions"]
    .transform(
        lambda s: s.shift(1).rolling(7, min_periods=3).sum()
    )
)

# Keep rows where both comparison windows exist
model_df = daily.dropna(
    subset=[
        "prior_7d_impressions",
        "future_7d_impressions"
    ]
).copy()

# Binary target:
# 1 = future 7-day impressions are below 80% of prior 7-day impressions
model_df["decline_label"] = (
    model_df["future_7d_impressions"]
    < 0.80 * model_df["prior_7d_impressions"]
).astype(int)

print("Modeling rows:", len(model_df))
print("Positive decline cases:", int(model_df["decline_label"].sum()))
print("Negative/non-decline cases:", int((model_df["decline_label"] == 0).sum()))

display(
    model_df[
        [
            "client_hash_id",
            "content_id",
            "report_date",
            "prior_7d_impressions",
            "future_7d_impressions",
            "decline_label"
        ]
    ].head()
)

Modeling rows: 7874092
Positive decline cases: 1378591
Negative/non-decline cases: 6495501


,client_hash_id,content_id,report_date,prior_7d_impressions,future_7d_impressions,decline_label
3,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,2026-03-04,0.0,0.0,0
4,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,2026-03-05,0.0,0.0,0
5,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,2026-03-06,0.0,0.0,0
6,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,2026-03-07,0.0,0.0,0
7,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,2026-03-08,0.0,0.0,0


In [12]:
# Create leakage-safe model features.
# The future impression column is deliberately NOT used as a feature.

feature_columns = [
    "prior_7d_impressions"
]

X = model_df[feature_columns].copy()
y = model_df["decline_label"].copy()

print("Features:", feature_columns)
print("Target:", "decline_label")
print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nClass distribution:")
print(y.value_counts(normalize=True).rename("proportion"))

Features: ['prior_7d_impressions']
Target: decline_label
X shape: (7874092, 1)
y shape: (7874092,)

Class distribution:
decline_label
0    0.824921
1    0.175079
Name: proportion, dtype: float64


In [13]:
# Reproducible stratified train/test split

if y.nunique() < 2:
    raise ValueError("The target contains only one class; a classifier cannot be trained.")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training positive rate:", round(y_train.mean(), 4))
print("Test positive rate:", round(y_test.mean(), 4))

Training rows: 6299273
Test rows: 1574819
Training positive rate: 0.1751
Test positive rate: 0.1751


In [14]:
# Majority-class baseline

majority_class = y_train.mode()[0]

baseline_predictions = np.full(
    shape=len(y_test),
    fill_value=majority_class
)

baseline_metrics = {
    "accuracy": accuracy_score(y_test, baseline_predictions),
    "precision": precision_score(
        y_test, baseline_predictions, zero_division=0
    ),
    "recall": recall_score(
        y_test, baseline_predictions, zero_division=0
    ),
    "f1": f1_score(
        y_test, baseline_predictions, zero_division=0
    )
}

print("Baseline metrics:")
print(pd.DataFrame([baseline_metrics]))

Baseline metrics:
   accuracy  precision  recall   f1
0  0.824921        0.0     0.0  0.0


In [15]:
# Logistic Regression pipeline
# Imputation + scaling + classifier

model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

model.fit(X_train, y_train)

model_predictions = model.predict(X_test)

model_metrics = {
    "accuracy": accuracy_score(y_test, model_predictions),
    "precision": precision_score(
        y_test, model_predictions, zero_division=0
    ),
    "recall": recall_score(
        y_test, model_predictions, zero_division=0
    ),
    "f1": f1_score(
        y_test, model_predictions, zero_division=0
    )
}

print("Logistic Regression metrics:")
print(pd.DataFrame([model_metrics]))

Logistic Regression metrics:
   accuracy  precision   recall        f1
0  0.824328   0.438884  0.01215  0.023646


In [16]:
# Compare baseline and model on exactly the same test set

comparison = pd.DataFrame(
    [
        {"method": "Majority-class baseline", **baseline_metrics},
        {"method": "Logistic Regression", **model_metrics}
    ]
)

display(comparison.round(4))

,method,accuracy,precision,recall,f1
0,Majority-class baseline,0.8249,0.0000,0.0000,0.0000
1,Logistic Regression,0.8243,0.4389,0.0122,0.0236


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The main error types are false positives and false negatives. A false positive means the model flags a content item as likely to decline when the measured future window does not meet the decline definition. A false negative means the model misses an observed decline.

The model should therefore be treated as a prioritization signal rather than an automatic decision rule. Its predictions are most useful for deciding which content items deserve human review.

In [17]:
# Confusion matrix

cm = confusion_matrix(y_test, model_predictions)

cm_df = pd.DataFrame(
    cm,
    index=["Observed 0", "Observed 1"],
    columns=["Predicted 0", "Predicted 1"]
)

display(cm_df)

,Predicted 0,Predicted 1
Observed 0,1294818,4283
Observed 1,272368,3350


In [18]:
# Detailed classification report

print(
    classification_report(
        y_test,
        model_predictions,
        digits=3,
        zero_division=0
    )
)

              precision    recall  f1-score   support

           0      0.826     0.997     0.903   1299101
           1      0.439     0.012     0.024    275718

    accuracy                          0.824   1574819
   macro avg      0.633     0.504     0.464   1574819
weighted avg      0.758     0.824     0.749   1574819



In [19]:
# Error analysis: show representative false positives and false negatives

error_df = model_df.loc[
    X_test.index,
    [
        "client_hash_id",
        "content_id",
        "report_date",
        "prior_7d_impressions",
        "future_7d_impressions",
        "decline_label"
    ]
].copy()

error_df["predicted_label"] = model_predictions

error_df["error_type"] = np.select(
    [
        (error_df["decline_label"] == 0) &
        (error_df["predicted_label"] == 1),

        (error_df["decline_label"] == 1) &
        (error_df["predicted_label"] == 0)
    ],
    [
        "false_positive",
        "false_negative"
    ],
    default="correct"
)

print("Error counts:")
print(error_df["error_type"].value_counts())

display(
    error_df[
        error_df["error_type"] != "correct"
    ].head(20)
)

Error counts:
error_type
correct           1298168
false_negative     272368
false_positive       4283
Name: count, dtype: int64


,client_hash_id,content_id,report_date,prior_7d_impressions,future_7d_impressions,decline_label,predicted_label,error_type
5707010,client_62f4a7e64f5e0096,content_b77f820e9102b191,2026-03-08,1226.0,755.0,1,0,false_negative
9505433,client_fef1a8f436438636,content_6ae2e8781f989363,2026-03-24,240.0,167.0,1,0,false_negative
9618283,client_fef1a8f436438636,content_c151b3cdd91f2101,2026-03-27,70.0,26.0,1,0,false_negative
1378400,client_20259bd6705d81d4,content_55bddcb5c34fe169,2026-03-21,472.0,238.0,1,0,false_negative
6383242,client_73cda7b4e4f265ea,content_0ba8117ce8356f45,2026-03-24,11.0,4.0,1,0,false_negative
3313665,client_3ffa76342f366962,content_55830416a2ad60bf,2026-03-05,1.0,0.0,1,0,false_negative
5368779,client_62f4a7e64f5e0096,content_462969d6c92bfa46,2026-03-22,228.0,106.0,1,0,false_negative
1940198,client_23a62021009f63c4,content_b31bef2fcaafd153,2026-03-26,159.0,61.0,1,0,false_negative
9558043,client_fef1a8f436438636,content_92f5681d8cea3e07,2026-03-28,709.0,107.0,1,0,false_negative
1640582,client_2094c6eb080311d5,content_fb0f3ac90e7de48d,2026-03-15,55.0,21.0,1,0,false_negative


In [20]:
# Simple interpretation check

print("Model interpretation:")
print(
    f"The model achieved an F1 score of "
    f"{model_metrics['f1']:.3f} on the held-out test set."
)

print(
    f"The majority-class baseline achieved an F1 score of "
    f"{baseline_metrics['f1']:.3f}."
)

if model_metrics["f1"] > baseline_metrics["f1"]:
    print(
        "Observed result: the model improved F1 over the majority-class baseline "
        "on this development split."
    )
else:
    print(
        "Observed result: the model did not improve F1 over the majority-class "
        "baseline on this development split."
    )

print(
    "This is directional model evaluation and does not establish causation "
    "or production performance."
)

Model interpretation:
The model achieved an F1 score of 0.024 on the held-out test set.
The majority-class baseline achieved an F1 score of 0.000.
Observed result: the model improved F1 over the majority-class baseline on this development split.
This is directional model evaluation and does not establish causation or production performance.


## Self-check

Before you submit, confirm each line honestly:

- [ yes] Every section above is filled — markdown thinking AND the code that backs it
- [ yes] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ yes] No client names, URLs, or private queries anywhere
- [ yes] My claims use careful words: observed, measured, directional, decision-support
- [ yes] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.